# FedLite-STGCN: Federated Financial Risk Management

**Implementation aligned with the supplied methodology**: Dirichlet Non-IID client partitioning, robust temporal normalization, sparse local financial graphs, depthwise-style graph aggregation + lightweight temporal state unit, drift-triggered local updates, FedAvg aggregation, and risk score/category prediction.

> **Data note:** The supplied document names public financial datasets but does not include download URLs, target-column definitions, or actual data files. This notebook therefore supports a local CSV (including IEEE-CIS-style transaction CSVs) and includes a clearly marked synthetic smoke-test fallback so the pipeline can be executed end-to-end. For meaningful research results, set `DATA_PATH` to your real labeled dataset and configure `TARGET_COLUMN`. The synthetic fallback is only for checking code execution, not for reporting results.

## 1. Install/import dependencies

In [ ]:
# If needed in Colab, uncomment:
# !pip -q install numpy pandas scikit-learn matplotlib torch

import os, glob, copy, random, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, classification_report)
from sklearn.neighbors import NearestNeighbors

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Configure and load the dataset

In [ ]:
# Set this to the path of your downloaded CSV.
# Examples:
# DATA_PATH = "/content/train_transaction.csv"  # IEEE-CIS
# DATA_PATH = "/content/financial_risk.csv"     # your own labeled CSV
DATA_PATH = ""

# Set explicitly if known (recommended). IEEE-CIS train_transaction.csv uses isFraud.
TARGET_COLUMN = None
TIME_COLUMN = None            # optional; e.g. "TransactionDT"
ID_COLUMNS = []               # optional columns to exclude, e.g. ["TransactionID"]
MAX_ROWS = 60000              # lower this for limited RAM; set None to use all rows
USE_SYNTHETIC_IF_MISSING = True

def make_synthetic_data(n=6000, seed=SEED):
    """Execution-only demo data; replace with real labeled data for research."""
    rng = np.random.default_rng(seed)
    amount = rng.lognormal(3.1, 1.0, n)
    hour = rng.integers(0, 24, n)
    velocity = rng.poisson(2.5, n)
    device_risk = rng.beta(1.5, 5, n)
    account_age = rng.integers(1, 1500, n)
    # Synthetic label signal, deliberately noisy
    logits = -4.0 + 0.0015*amount + 0.18*velocity + 2.8*device_risk + 0.035*((hour < 5).astype(float))
    prob = 1/(1+np.exp(-logits))
    y = rng.binomial(1, prob)
    return pd.DataFrame({
        "TransactionAmt": amount, "hour": hour, "velocity_24h": velocity,
        "device_risk": device_risk, "account_age_days": account_age, "isFraud": y
    })

if DATA_PATH and os.path.isfile(DATA_PATH):
    df = pd.read_csv(DATA_PATH, nrows=MAX_ROWS)
    print("Loaded:", DATA_PATH, "| shape:", df.shape)
elif USE_SYNTHETIC_IF_MISSING:
    df = make_synthetic_data()
    print("WARNING: using synthetic smoke-test data. Set DATA_PATH to real data for valid results.")
else:
    raise FileNotFoundError("Set DATA_PATH to your labeled CSV file.")

display(df.head())
print(df.shape)

## 3. Prepare features, target, and temporal order

In [ ]:
# Auto-detect a common fraud/risk target only if TARGET_COLUMN was not supplied.
if TARGET_COLUMN is None:
    candidates = ["isFraud", "fraud", "label", "target", "risk", "Risk", "Class", "class"]
    found = [c for c in candidates if c in df.columns]
    if not found:
        raise ValueError("Target column not found. Set TARGET_COLUMN to the label column in your CSV.")
    TARGET_COLUMN = found[0]

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"TARGET_COLUMN={TARGET_COLUMN!r} not in columns.")

# Drop rows with missing target; encode categorical target labels.
df = df.loc[df[TARGET_COLUMN].notna()].copy()
target_encoder = LabelEncoder()
y_all = target_encoder.fit_transform(df[TARGET_COLUMN].astype(str))
classes = np.unique(y_all)
if len(classes) < 2:
    raise ValueError("Classification requires at least two target classes.")

exclude = set([TARGET_COLUMN] + ID_COLUMNS)
if TIME_COLUMN and TIME_COLUMN in df.columns:
    # Preserve temporal order; time itself may also be included as a feature below.
    df = df.sort_values(TIME_COLUMN, kind="stable").reset_index(drop=True)
    y_all = target_encoder.fit_transform(df[TARGET_COLUMN].astype(str))
    exclude.add(TIME_COLUMN)

Xdf = df.drop(columns=[c for c in exclude if c in df.columns]).copy()
# Convert booleans to numeric and categorical columns to one-hot encoding.
for c in Xdf.columns:
    if Xdf[c].dtype == "bool":
        Xdf[c] = Xdf[c].astype(int)
Xdf = pd.get_dummies(Xdf, dummy_na=True)
Xdf = Xdf.replace([np.inf, -np.inf], np.nan)

# Keep numeric columns only after encoding, then impute and robust-scale.
Xraw = Xdf.to_numpy(dtype=np.float32)
Xraw = SimpleImputer(strategy="median").fit_transform(Xraw).astype(np.float32)
Xraw = RobustScaler().fit_transform(Xraw).astype(np.float32)
Xraw = np.nan_to_num(Xraw, nan=0.0, posinf=0.0, neginf=0.0)

# Keep a manageable feature dimension for lightweight graph training.
MAX_FEATURES = 64
if Xraw.shape[1] > MAX_FEATURES:
    variances = Xraw.var(axis=0)
    keep = np.argsort(variances)[-MAX_FEATURES:]
    Xraw = Xraw[:, keep]

# Train/test split. For time-sorted data use chronological split; otherwise stratified random split.
indices = np.arange(len(y_all))
if TIME_COLUMN and TIME_COLUMN in df.columns:
    split = int(0.8 * len(indices))
    train_idx, test_idx = indices[:split], indices[split:]
else:
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, random_state=SEED, stratify=y_all
    )

X_train, X_test = Xraw[train_idx], Xraw[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]
print("Target:", TARGET_COLUMN, "| classes:", list(target_encoder.classes_))
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class counts:", dict(zip(*np.unique(y_train, return_counts=True))))

## 4. Federated client formation with Dirichlet Non-IID partitioning

In [ ]:
NUM_CLIENTS = 3
DIRICHLET_ALPHA = 0.5   # smaller => more heterogeneous client label distributions

def dirichlet_partition(y, n_clients=3, alpha=0.5, seed=42):
    rng = np.random.default_rng(seed)
    client_indices = [[] for _ in range(n_clients)]
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)
        proportions = rng.dirichlet(np.repeat(alpha, n_clients))
        cuts = (np.cumsum(proportions) * len(cls_idx)).astype(int)[:-1]
        splits = np.split(cls_idx, cuts)
        for cid, split_idx in enumerate(splits):
            client_indices[cid].extend(split_idx.tolist())
    for part in client_indices:
        rng.shuffle(part)
    # Ensure every client has data; rare edge cases fall back to balanced split.
    if any(len(p) == 0 for p in client_indices):
        perm = rng.permutation(len(y))
        client_indices = [p.tolist() for p in np.array_split(perm, n_clients)]
    return [np.asarray(p, dtype=int) for p in client_indices]

client_parts = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, SEED)
for i, part in enumerate(client_parts):
    print(f"Client {i+1}: n={len(part)}, class distribution={dict(zip(*np.unique(y_train[part], return_counts=True)))}")

## 5. Temporal windows and sparse local financial graphs

In [ ]:
WINDOW_SIZE = 16
WINDOW_STRIDE = 8
TOP_K_NEIGHBORS = 4
MAX_WINDOWS_PER_CLIENT = 1200  # computational safeguard; None to use all windows

def make_windows(X, y, window_size=16, stride=8, max_windows=None):
    """Create ordered windows. Window label is the maximum risk label in the window."""
    if len(X) == 0:
        return []
    if len(X) < window_size:
        pad = window_size - len(X)
        X = np.pad(X, ((0,pad),(0,0)), mode="edge")
        y = np.pad(y, (0,pad), mode="edge")
    starts = list(range(0, max(1, len(X)-window_size+1), stride))
    if not starts:
        starts = [0]
    if max_windows and len(starts) > max_windows:
        # deterministic sampling preserves broad temporal coverage
        chosen = np.linspace(0, len(starts)-1, max_windows).astype(int)
        starts = [starts[i] for i in chosen]
    out = []
    for s in starts:
        xx = X[s:s+window_size]
        yy = y[s:s+window_size]
        if len(xx) < window_size:
            continue
        out.append((xx.astype(np.float32), int(np.max(yy))))
    return out

def sparse_knn_adjacency(x, k=4):
    """Symmetric sparse kNN adjacency with self-loops, normalized by node degree."""
    n = len(x)
    if n <= 1:
        return np.eye(n, dtype=np.float32)
    k_eff = min(k+1, n)
    nbrs = NearestNeighbors(n_neighbors=k_eff, metric="euclidean").fit(x)
    _, inds = nbrs.kneighbors(x)
    A = np.zeros((n,n), dtype=np.float32)
    for i, row in enumerate(inds):
        for j in row[1:]:
            A[i,j] = 1.0
            A[j,i] = 1.0
    np.fill_diagonal(A, 1.0)
    deg = A.sum(axis=1)
    inv = 1.0 / np.sqrt(np.maximum(deg, 1.0))
    return (inv[:,None] * A * inv[None,:]).astype(np.float32)

client_windows = []
for cid, part in enumerate(client_parts):
    wins = make_windows(X_train[part], y_train[part], WINDOW_SIZE, WINDOW_STRIDE, MAX_WINDOWS_PER_CLIENT)
    graphs = [(x, sparse_knn_adjacency(x, TOP_K_NEIGHBORS), label) for x, label in wins]
    client_windows.append(graphs)
    print(f"Client {cid+1}: {len(graphs)} local temporal graphs")

## 6. FedLite-STGCN model

In [ ]:
class GraphWindowDataset(Dataset):
    def __init__(self, graphs):
        self.graphs = graphs
    def __len__(self):
        return len(self.graphs)
    def __getitem__(self, idx):
        x, a, y = self.graphs[idx]
        return torch.tensor(x), torch.tensor(a), torch.tensor(y, dtype=torch.long)

class FedLiteSTGCN(nn.Module):
    """Sparse-graph-inspired spatial aggregation + lightweight gated temporal state unit."""
    def __init__(self, input_dim, hidden=32, num_classes=2):
        super().__init__()
        self.node_proj = nn.Linear(input_dim, hidden)
        # Depthwise-style channel mixing: per-channel spatial aggregation followed by pointwise projection.
        self.depthwise_scale = nn.Parameter(torch.ones(hidden))
        self.pointwise = nn.Linear(hidden, hidden)
        self.norm = nn.LayerNorm(hidden)
        self.temporal = nn.GRU(input_size=hidden, hidden_size=hidden, num_layers=1, batch_first=True)
        self.state_gate = nn.Sequential(nn.Linear(hidden*2, hidden), nn.Sigmoid())
        self.head = nn.Sequential(nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(0.15),
                                  nn.Linear(hidden//2, num_classes))
    def forward(self, x, adj):
        # x: [B,T,F], adj: [B,T,T]
        h = torch.relu(self.node_proj(x))
        spatial = torch.bmm(adj, h * self.depthwise_scale)
        h = torch.relu(self.norm(self.pointwise(spatial) + h))
        seq, _ = self.temporal(h)
        # Lightweight continual state approximation: gated blend of latest state and pooled context.
        pooled = seq.mean(dim=1)
        latest = seq[:, -1, :]
        gate = self.state_gate(torch.cat([latest, pooled], dim=-1))
        state = gate * latest + (1.0 - gate) * pooled
        return self.head(state), state

def build_model():
    return FedLiteSTGCN(input_dim=X_train.shape[1], hidden=32, num_classes=len(classes)).to(DEVICE)

model = build_model()
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Local training, drift gate, and federated FedAvg

In [ ]:
LOCAL_EPOCHS = 2
BATCH_SIZE = 32
ROUNDS = 3
LEARNING_RATE = 1e-3
DRIFT_THRESHOLD = 0.20

def local_train(global_state, graphs, epochs=2):
    local_model = build_model()
    local_model.load_state_dict(global_state)
    if len(graphs) == 0:
        return copy.deepcopy(global_state), 0.0
    loader = DataLoader(GraphWindowDataset(graphs), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    opt = torch.optim.Adam(local_model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()
    local_model.train()
    total_loss, batches = 0.0, 0
    for _ in range(epochs):
        for xb, ab, yb in loader:
            xb, ab, yb = xb.to(DEVICE), ab.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits, _ = local_model(xb, ab)
            loss = loss_fn(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local_model.parameters(), 2.0)
            opt.step()
            total_loss += float(loss.item()); batches += 1
    return copy.deepcopy(local_model.state_dict()), total_loss / max(batches, 1)

def weighted_fedavg(states, weights):
    total = float(sum(weights))
    if total <= 0:
        return copy.deepcopy(states[0])
    out = {}
    for key in states[0].keys():
        out[key] = sum(states[i][key] * (weights[i] / total) for i in range(len(states)))
    return out

global_model = build_model()
global_state = copy.deepcopy(global_model.state_dict())
previous_client_states = [None] * NUM_CLIENTS
round_losses = []
round_updates = []

for rnd in range(ROUNDS):
    local_states, local_weights, losses, updated_clients = [], [], [], 0
    for cid, graphs in enumerate(client_windows):
        candidate_state, local_loss = local_train(global_state, graphs, LOCAL_EPOCHS)
        # Drift gate based on average feature-state change from a small local sample.
        drift = 1.0
        if graphs:
            global_model.load_state_dict(global_state)
            global_model.eval()
            x0, a0, _ = graphs[0]
            with torch.no_grad():
                _, current_state = global_model(
                    torch.tensor(x0[None], dtype=torch.float32, device=DEVICE),
                    torch.tensor(a0[None], dtype=torch.float32, device=DEVICE))
            if previous_client_states[cid] is not None:
                drift = float(torch.norm(current_state.cpu() - previous_client_states[cid]) /
                              (torch.norm(previous_client_states[cid]) + 1e-8))
            previous_client_states[cid] = current_state.detach().cpu()
        # First round always updates; later rounds only accept candidate if drift exceeds threshold.
        if rnd == 0 or drift >= DRIFT_THRESHOLD:
            local_states.append(candidate_state)
            local_weights.append(max(len(graphs), 1))
            losses.append(local_loss)
            updated_clients += 1
        else:
            # Stable client reuses its current/global state, so it contributes no new update.
            continue
    if local_states:
        global_state = weighted_fedavg(local_states, local_weights)
        global_model.load_state_dict(global_state)
    avg_loss = float(np.mean(losses)) if losses else float("nan")
    round_losses.append(avg_loss)
    round_updates.append(updated_clients)
    print(f"Round {rnd+1}/{ROUNDS} | participating updates={updated_clients}/{NUM_CLIENTS} | mean local loss={avg_loss:.4f}")

model = build_model()
model.load_state_dict(global_state)

## 8. Evaluation with local test-stream windows

In [ ]:
test_graphs = [(x, sparse_knn_adjacency(x, TOP_K_NEIGHBORS), label)
               for x, label in make_windows(X_test, y_test, WINDOW_SIZE, WINDOW_STRIDE, MAX_WINDOWS_PER_CLIENT)]
if len(test_graphs) == 0:
    raise RuntimeError("No test windows were created; check dataset size/window settings.")

test_loader = DataLoader(GraphWindowDataset(test_graphs), batch_size=BATCH_SIZE, shuffle=False)
model.eval()
all_true, all_pred, all_prob = [], [], []
with torch.no_grad():
    for xb, ab, yb in test_loader:
        xb, ab = xb.to(DEVICE), ab.to(DEVICE)
        logits, _ = model(xb, ab)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_true.extend(yb.numpy().tolist())
        all_pred.extend(preds.cpu().numpy().tolist())
        all_prob.extend(probs.cpu().numpy().tolist())

all_true = np.asarray(all_true); all_pred = np.asarray(all_pred); all_prob = np.asarray(all_prob)
acc = accuracy_score(all_true, all_pred)
prec, rec, f1, _ = precision_recall_fscore_support(all_true, all_pred, average="weighted", zero_division=0)
print(f"Window-level Accuracy : {acc:.4f}")
print(f"Weighted Precision    : {prec:.4f}")
print(f"Weighted Recall       : {rec:.4f}")
print(f"Weighted F1-score     : {f1:.4f}")
try:
    if len(classes) == 2:
        auc = roc_auc_score(all_true, all_prob[:,1])
    else:
        auc = roc_auc_score(all_true, all_prob, multi_class="ovr", average="weighted")
    print(f"ROC-AUC               : {auc:.4f}")
except Exception as e:
    print("ROC-AUC unavailable:", e)

print("\nClassification report (window-level):")
print(classification_report(all_true, all_pred, labels=np.arange(len(classes)),
                            target_names=[str(c) for c in target_encoder.classes_], zero_division=0))
print("Confusion matrix:\n", confusion_matrix(all_true, all_pred, labels=np.arange(len(classes))))

## 9. Training and federated participation plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(1, len(round_losses)+1), round_losses, marker="o")
axes[0].set_title("Mean local training loss")
axes[0].set_xlabel("Federated round"); axes[0].set_ylabel("Loss"); axes[0].grid(True, alpha=.3)
axes[1].bar(np.arange(1, len(round_updates)+1), round_updates)
axes[1].set_title("Clients contributing updates")
axes[1].set_xlabel("Federated round"); axes[1].set_ylabel("Updated clients"); axes[1].grid(True, axis="y", alpha=.3)
plt.tight_layout(); plt.show()

## 10. Risk score and category prediction for a new transaction window

In [ ]:
def predict_risk_window(feature_rows, model=model, threshold=0.5):
    """feature_rows must be an array [time_steps, n_features] already transformed
    with the same feature encoding, imputer, and scaler used during training."""
    arr = np.asarray(feature_rows, dtype=np.float32)
    if arr.ndim != 2 or arr.shape[1] != X_train.shape[1]:
        raise ValueError(f"Expected [T, {X_train.shape[1]}] feature array.")
    if len(arr) < WINDOW_SIZE:
        arr = np.pad(arr, ((WINDOW_SIZE-len(arr),0),(0,0)), mode="edge")
    arr = arr[-WINDOW_SIZE:]
    adj = sparse_knn_adjacency(arr, TOP_K_NEIGHBORS)
    model.eval()
    with torch.no_grad():
        logits, _ = model(torch.tensor(arr[None], device=DEVICE),
                          torch.tensor(adj[None], device=DEVICE))
        prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
    cls_idx = int(np.argmax(prob))
    return {
        "risk_score": float(prob[cls_idx]),
        "risk_class": str(target_encoder.inverse_transform([cls_idx])[0]),
        "class_probabilities": {str(target_encoder.classes_[i]): float(prob[i]) for i in range(len(prob))}
    }

# Demonstration on one held-out test window (model's predicted class probability).
demo_x, demo_a, _ = test_graphs[0]
demo_result = predict_risk_window(demo_x)
print(demo_result)

## Implementation notes and limitations

- **Privacy simulation:** raw records remain in local client partitions in this notebook; only model parameters are aggregated. This is a single-process simulation, not production secure aggregation.
- **Graph construction:** each temporal window is represented as nodes with a symmetric k-nearest-neighbor graph, retaining only a sparse set of edges.
- **Continual state/drift:** the lightweight GRU state and drift gate are practical approximations of the described state-space and risk-drift update concepts. They are not a formal state-space model or a statistically calibrated drift detector.
- **Label semantics:** window labels use `max(label)` (a window is risky if any transaction in it is risky). Change this if your research defines risk differently.
- **No fabricated results:** reported metrics are calculated from the dataset actually loaded. Do not report synthetic smoke-test results as real-world performance.
- **Scale:** for very large datasets, lower `MAX_ROWS`, `MAX_WINDOWS_PER_CLIENT`, `WINDOW_SIZE`, or `ROUNDS` to fit available RAM/GPU memory.